# 07 · Does legal data help? (Study 5, GPU notebook)

You will train and score every system in the pre-registered study `lab/legal/DESIGN.md`, test its four
hypotheses, choose a model with the pre-stated rule, and get charts, an error analysis and `REPORT.md`.

| Step | What happens | Time on Kaggle (2 × T4) |
| --- | --- | --- |
| 0 | set up, load the legal-web data you built on your Mac, check it wasn't changed | 5 min |
| 1 | score the baselines (rules, Presidio, two Turkish NER models, GLiNER) | 30 min |
| 2 | train 5 systems × 5 seeds of BERTurk (resumable) | about 4 h |
| 3 | choose `trust` on dev for each run, then open the test sets once | runs alongside step 2 |
| 4 | test H1–H4, apply the selection rule | 2 min |
| 5 | charts, error analysis, `REPORT.md`, download | 5 min |

**Where to run it.** Kaggle: *Create → New notebook → File → Import notebook → GitHub* and paste
`https://github.com/fikrifirat/katip/blob/legal-study/lab/legal/07_legal_study.ipynb`. In the right panel,
set *Accelerator = GPU T4 × 2* and *Internet = on*. Colab works too (one T4, so step 2 takes about 8 h; it
resumes after a disconnect if you keep `WORK` on Google Drive).

**Your data.** On your Mac, after the crawl: `python lab/legal/build_dataset.py`, then
`cd lab/data && zip -r legal_web.zip legal_web -x 'legal_web/raw.jsonl' 'legal_web/*.log'`. Upload that zip as
a **private** Kaggle dataset named `katip-legal-web` (*Add data → Upload*). It holds only slot-filled
decisions (every name, ID and address in them is synthetic) and AYM judgments with the names the court
itself published. The raw crawl stays on your Mac.

A failed check (`assert`) stops the notebook, so you never get numbers you can't trust.

In [ ]:
# Step 0a · get the code (branch legal-study) and the packages
import os, subprocess, sys
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
REPO = f"{WORK}/katip"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", "legal-study", "--depth", "1", "https://github.com/fikrifirat/katip", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/lab/requirements.txt",
                "presidio-analyzer", "spacy", "gliner", "datasets",
                "https://github.com/explosion/spacy-models/releases/download/xx_ent_wiki_sm-3.8.0/xx_ent_wiki_sm-3.8.0-py3-none-any.whl"], check=True)
os.chdir(f"{REPO}/lab"); sys.path[:0] = [f"{REPO}/lab", f"{REPO}/lab/legal"]
print(subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=iso"], capture_output=True, text=True).stdout)

In [ ]:
# Step 0b · GPUs
import torch
GPUS = torch.cuda.device_count()
print(GPUS, "GPU(s):", [torch.cuda.get_device_name(i) for i in range(GPUS)])
assert GPUS >= 1, "Turn on a GPU accelerator in the notebook settings."

In [ ]:
# Step 0c · the legal-web data, unchanged since build_dataset.py wrote it
import glob, json, shutil, zipfile
import katiplab as K
dst = K.DATA / "legal_web"
if not (dst / "manifest.json").exists():
    z = (glob.glob("/kaggle/input/*/legal_web.zip") + glob.glob("/kaggle/input/**/legal_web.zip", recursive=True) + glob.glob(f"{WORK}/legal_web.zip"))
    if z: zipfile.ZipFile(z[0]).extractall(K.DATA)
    else:  # already unzipped by Kaggle
        src = (glob.glob("/kaggle/input/*/legal_web") + glob.glob("/kaggle/input/*/*/legal_web"))[0]; shutil.copytree(src, dst)
import study as S
m = S.check_hashes()
print(json.dumps(m["counts"], indent=1, ensure_ascii=False)); print(json.dumps(m["by_type"], ensure_ascii=False))
assert m["counts"]["test"]["docs"] >= 300 and m["counts"]["aym_names"]["docs"] >= 50, "too little test data: let the crawl run longer"

## The pre-registration

Hypotheses, metrics, splits and the selection rule were committed **before** any of this ran
(`lab/legal/DESIGN.md`, amendment A1 included). This cell prints the commits that touched it, so the
report can cite them. Don't edit DESIGN.md from here.

In [ ]:
print(subprocess.run(["git", "log", "--format=%h %ad %s", "--date=iso", "--", "legal/DESIGN.md"], capture_output=True, text=True).stdout or "(shallow clone: see GitHub history)")

## Step 1 · Baselines

Rules only (Katip v0.2, Presidio-style), vanilla Presidio + spaCy, two off-the-shelf Turkish NER models and
GLiNER (zero-shot PII). None of them is tuned on our data. TLNER (CC BY-NC 4.0) is downloaded here as an
external test set; it is never used for training.

In [ ]:
try: S.get_tlner(); print("TLNER test ready")
except Exception as e: print("TLNER skipped:", e)
import baselines as BL
S.run_system("Rules")
for key, (name, make) in BL.BASELINES.items():
    tag = {"presidio": "R1-Presidio", "savasy": "X1-savasy", "akdeniz": "X2-xlmr", "gliner": "X3-gliner"}[key]
    if (S.OUT / f"{tag}.json").exists(): continue
    try: det = make()
    except Exception as e: print("skip", name, e); continue
    S.run_system(tag, detector=det, keep_rows=True)
    S.run_system(tag + "+rules", detector=BL.with_rules(det), keep_rows=True)
    print("done", name)

## Step 2 · Train 5 systems × 5 seeds

| System | Training data | Flags |
| --- | --- | --- |
| B0 | synthetic (v1 recipe), the **baseline** | |
| M1 | + 18k real Wikipedia sentences (v2 recipe) | `--real-text` |
| M2 | M1 + slot-filled court decisions | `--real-text --legal` |
| M2-vol | M1 + the same number of extra synthetic docs (volume control) | `--real-text --extra-synth match` |
| M2-noDAN | M2 without Danıştay (unseen-court test) | `--real-text --legal --exclude-court danistay` |

Each run writes `runs/legal/<system>_s<seed>/`. Finished runs are skipped, so you can re-run this cell after
a disconnect. To save disk, only the int8 ONNX file is kept, except for seed 0 of every system.

In [ ]:
import time
from pathlib import Path
RUNS_DIR = K.RUNS / "legal"; RUNS_DIR.mkdir(parents=True, exist_ok=True)
TRAIN = f"{REPO}/ml/berturk/train_berturk.py"; LEGAL = str(K.DATA / "legal_web" / "train.jsonl")
FLAGS = {"B0": [], "M1": ["--real-text"], "M2": ["--real-text", "--legal", LEGAL],
         "M2-vol": ["--real-text", "--extra-synth", "match", "--extra-synth-match", LEGAL],
         "M2-noDAN": ["--real-text", "--legal", LEGAL, "--exclude-court", "danistay"]}
JOBS = [(s, seed) for seed in S.SEEDS for s in FLAGS]          # seed-major: every system gets seed 0 first
def done(name): return (RUNS_DIR / name / "onnx" / "model_quantized.onnx").exists()
def launch(system, seed, gpu):
    name = f"{system}_s{seed}"; out = RUNS_DIR / name
    cmd = [sys.executable, TRAIN, "--seed", str(seed), "--out", str(out), *FLAGS[system]]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    return name, subprocess.Popen(cmd, env=env, stdout=open(RUNS_DIR / f"{name}.log", "w"), stderr=subprocess.STDOUT)
def tidy(name):
    out = RUNS_DIR / name
    if not name.endswith("_s0"):
        for f in (out / "model.safetensors", out / "onnx" / "model.onnx"):
            if f.exists(): f.unlink()

In [ ]:
# Smoke test first: one tiny run, 3 minutes. It must finish and score above zero.
smoke = RUNS_DIR / "smoke"
if not (smoke / "onnx" / "model_quantized.onnx").exists():
    r = subprocess.run([sys.executable, TRAIN, "--limit", "200", "--epochs", "1", "--out", str(smoke), "--real-text", "--legal", LEGAL],
                       capture_output=True, text=True)
    print(r.stdout[-1500:]); assert r.returncode == 0, r.stderr[-2000:]
assert (smoke / "onnx" / "model_quantized.onnx").exists()

In [ ]:
# The full matrix, one job per GPU. Re-run this cell any time; it continues where it stopped.
queue = [j for j in JOBS if not done(f"{j[0]}_s{j[1]}")]
running = {}
print(len(queue), "runs to go")
while queue or running:
    for gpu in range(GPUS):
        if gpu not in running and queue:
            s, seed = queue.pop(0); running[gpu] = launch(s, seed, gpu); print(time.strftime("%H:%M"), "start", running[gpu][0], "on GPU", gpu)
    time.sleep(30)
    for gpu, (name, p) in list(running.items()):
        if p.poll() is not None:
            ok = p.returncode == 0 and done(name)
            print(time.strftime("%H:%M"), "finished" if ok else "FAILED", name, "" if ok else (RUNS_DIR / f"{name}.log").read_text()[-800:])
            if ok: tidy(name)
            del running[gpu]

## Step 3 · Choose trust on dev, then open the test sets once

For each run: try `trust` ∈ {0.3, 0.5, 0.7, 0.9} on dev only (legal-web dev, synthetic held-out, Wiki dev)
and keep the setting that meets the hard limits (legal-dev precision ≥ 95%, held-out recall ≥ 99.5%) with
the highest legal-dev recall. Then score every test set once. The model is also scored alone (no rules), so
a gain can't hide in the merge step. You can run this cell while step 2 is still going (it scores on CPU).

In [ ]:
for s, seed in JOBS:
    name = f"{s}_s{seed}"
    if done(name) and not (S.OUT / f"{name}.json").exists():
        t = time.time(); r = S.run_system(name, model_dir=RUNS_DIR / name, seed=seed)
        lw = r["test"]["legal-web test"]["types"]["PERSON"]
        print(f"{name:12} trust {r['trust']}  legal-web test recall {lw['recall']:.3f} prec {lw['precision']:.3f}  ({time.time() - t:.0f}s)")

## Step 4 · The hypotheses

Each line is the pre-registered test. *PASS* needs the Holm-corrected p < 0.05 and the extra condition
written in DESIGN.md (precision for H3, rank agreement for H4).

In [ ]:
import pandas as pd
V = S.hypotheses()
rows = []
for h, what in [("H1", "synthetic scores overstate real performance"), ("H2", "real text closes the gap (M1 > B0)"),
                ("H3", "legal data helps on legal text (M2 ≥ M1 + 1 pt)"), ("H3-vol", "…and it's the domain, not the volume (M2 > M2-vol)"),
                ("H4", "the slot-filled legal test predicts real-name recall (±3 pt)")]:
    v = V.get(h, {}); hh = V["holm"].get(h, {})
    rows.append({"hypothesis": h, "claim": what, "p (Holm)": hh.get("p_holm"), "result": "PASS" if v.get("pass") else ("FAIL" if v else "not run")})
display(pd.DataFrame(rows))
for h in ("H2", "H3"):
    for s, d in V.get(h, {}).get("detail", {}).items():
        print(f"{h:6} {s:16} diff {100 * d['diff']:+.1f} pt  95% CI [{100 * d['ci'][0]:+.1f}, {100 * d['ci'][1]:+.1f}]")
if "H3-vol" in V:
    d = V["H3-vol"]["detail"]; print(f"H3-vol legal-web test diff {100 * d['diff']:+.1f} pt  95% CI [{100 * d['ci'][0]:+.1f}, {100 * d['ci'][1]:+.1f}]")
if "H3" in V: print("H3 precision change:", {k: f"{100 * v:+.2f} pt" for k, v in V["H3"]["precision_change"].items()},
                   "| Wiki non-inferiority:", V["H3"]["H3-wiki non-inferiority"])
if "H4" in V: print("H4 Kendall tau:", round(V["H4"]["kendall_tau"], 2), "| gaps:", {k: f"{100 * v:+.1f}" for k, v in V["H4"]["gap"].items()})

In [ ]:
best, cands = S.select_model()
display(pd.DataFrame(cands)); print("selected on dev:", best and best["system"])

## Step 5 · Charts, errors, report

In [ ]:
import numpy as np, matplotlib.pyplot as plt
R = S.load_results()
SETS = [s for s in ("blind (synthetic)", "wiki test", "TurkishLegalNER (filled)", "legal-web test", "AYM real names", "TLNER (external, NC)")
        if any(s in r["test"] for r in R.values())]
ORDER = ["R1-Presidio", "X1-savasy", "X2-xlmr", "X3-gliner", "Rules", "B0", "M1", "M2-vol", "M2-noDAN", "M2"]
def stats(name, s, typ="PERSON"):
    runs = S.system_runs(R, name) or ([R[name]] if name in R else [])
    runs = [r for r in runs if s in r["test"] and typ in r["test"][s]["types"]]
    if not runs: return None
    v = [r["test"][s]["types"][typ]["recall"] for r in runs]
    lo = np.mean([r["test"][s]["types"][typ]["recall_ci"][0] for r in runs]); hi = np.mean([r["test"][s]["types"][typ]["recall_ci"][1] for r in runs])
    return np.mean(v), lo, hi, v
names = [n for n in ORDER if stats(n, SETS[0]) or any(stats(n, s) for s in SETS)]
fig, axs = plt.subplots(1, len(SETS), figsize=(4.2 * len(SETS), 0.55 * len(names) + 1.6), sharey=True)
col = {"B0": "#eb6834", "M2": "#2a78d6"}
for ax, s in zip(np.atleast_1d(axs), SETS):
    for i, n in enumerate(names):
        st = stats(n, s); y = len(names) - 1 - i
        if not st: continue
        m, lo, hi, v = st; c = col.get(n, "#a3a29c")
        ax.plot([100 * lo, 100 * hi], [y, y], color=c, lw=2); ax.scatter([100 * x for x in v], [y] * len(v), s=10, color=c, alpha=.5, zorder=3)
        ax.plot(100 * m, y, "o", ms=8, color=c, mec="white", mew=1.5, zorder=4); ax.text(100 * m, y + .3, f"{100 * m:.1f}", ha="center", fontsize=8)
    ax.set_title(s, fontsize=10, loc="left"); ax.set_xlim(0, 103); ax.grid(axis="x", color="#eee"); [ax.spines[k].set_visible(False) for k in ("top", "right", "left")]
np.atleast_1d(axs)[0].set_yticks(range(len(names))); np.atleast_1d(axs)[0].set_yticklabels(names[::-1])
fig.suptitle("PERSON recall by system and test set (dots = seeds, bar = mean 95% CI). Orange = baseline B0, blue = M2", x=0.01, ha="left")
plt.tight_layout(); plt.savefig(S.OUT / "_recall_by_set.png", dpi=180, bbox_inches="tight"); plt.show()

In [ ]:
# Every PII type, system level (rules + model), on the legal-web test set: seed-mean recall / precision
tbl = {}
for n in names:
    runs = S.system_runs(R, n) or ([R[n]] if n in R else [])
    if not runs or "legal-web test" not in runs[0]["test"]: continue
    row = {}
    for t in S.SYSTEM_TYPES:
        v = [r["test"]["legal-web test"]["types"].get(t) for r in runs]
        if all(v): row[t] = f"{100 * np.mean([x['recall'] for x in v]):.1f} / {100 * np.nanmean([x['precision'] for x in v]):.1f}"
    row["leak rate %"] = f"{100 * np.mean([r['test']['legal-web test']['leak_rate'] for r in runs]):.2f}"
    row["clean docs %"] = f"{100 * np.mean([r['test']['legal-web test']['clean_doc_rate'] for r in runs]):.1f}"
    tbl[n] = row
display(pd.DataFrame(tbl).T)

In [ ]:
E = S.error_analysis()
display(pd.DataFrame({k: v["counts"] for k, v in E.items()}).fillna(0).astype(int).sort_index())
for k, ex in E.get("M2_s0", {}).get("examples", {}).items():
    print(f"\n{k}:"); [print("   …" + e.replace(chr(10), " ") + "…") for e in ex[:3]]

In [ ]:
# REPORT.md + a zip of everything in results/legal to download (Kaggle: Output panel)
lines = ["# Study 5 report: legal-domain data for Turkish PII detection", "",
         f"Pre-registration: lab/legal/DESIGN.md. Seeds per system: {V['seeds']}.", "",
         "| Hypothesis | p (Holm) | Result |", "| --- | --- | --- |"]
for r_ in rows: lines.append(f"| {r_['hypothesis']}: {r_['claim']} | {r_['p (Holm)'] if r_['p (Holm)'] is None else round(r_['p (Holm)'], 4)} | {r_['result']} |")
lines += ["", "## PERSON recall, seed mean [95% CI]", "", "| System | " + " | ".join(SETS) + " |", "| --- |" + " --- |" * len(SETS)]
for n in names:
    cells_ = []
    for s in SETS:
        st = stats(n, s); cells_.append(f"{100 * st[0]:.1f} [{100 * st[1]:.1f}–{100 * st[2]:.1f}]" if st else "")
    lines.append(f"| {n} | " + " | ".join(cells_) + " |")
lines += ["", f"Selected on dev by the pre-stated rule: **{best and best['system']}**.", "", "Charts: _recall_by_set.png. Errors: _errors.json."]
(S.OUT / "REPORT.md").write_text("\n".join(lines)); print("\n".join(lines))
shutil.make_archive(f"{WORK}/legal_results", "zip", S.OUT); print("download:", f"{WORK}/legal_results.zip")